# HoTHP Grid Search: Fast Decay + Slow Decay (Combined)

**Como rodar:** Runtime -> Change runtime type -> T4 GPU (ou L4)

---

## Objetivo

Notebook unificado que executa o grid search para **ambos** os cenarios
(decaimento rapido e lento) e produz:

1. Tabela comparativa no formato do paper (Table 1)
2. Heatmaps lado a lado (fast vs slow)
3. Graficos de linhas NLL por fator de extrapolacao
4. Teste de **Wilcoxon signed-rank** para validacao estatistica

A NLL eh computada sobre a **sequencia completa** (nao apenas a parte extrapolada).

**Tempo estimado:** ~4-5h numa T4 (2 processos x 4 train_lens x 3 fatores x 5 seeds x 2 modelos)

In [ ]:
# ============================================================================
# CELULA 1 — Instalacao
# ============================================================================

import os

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

!pip install omegaconf -q

# Fixes
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write("""from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel
from easy_tpp.model.torch_model.torch_thp import THP as TorchTHP
from easy_tpp.model.torch_model.torch_rothp import RoTHP as TorchRoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP as TorchHoTHP
""")

_hothp_path = 'ufc-easytpp/easy_tpp/model/torch_model/torch_hothp.py'
with open(_hothp_path, 'r') as f:
    code = f.read()
if 'from notebooks.' in code:
    code = code.replace(
        'from notebooks.Extrapolation_and_Attention_Analysis import attention_fixed\n', '')
    with open(_hothp_path, 'w') as f:
        f.write(code)

print('OK')

In [ ]:
# ============================================================================
# CELULA 2 — Imports e configuracao
# ============================================================================

import os, sys, math, random, hashlib, contextlib, time, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from scipy import stats
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

# ── Grid de busca ─────────────────────────────────────────────────────────

TRAIN_LENS     = [10, 20, 50, 100]
EXTRAP_FACTORS = [2, 5, 10]
N_SEEDS        = 5
EPOCHS         = 500
PATIENCE       = 30
BASE_SEED      = 42
NUM_TYPES      = 2
PAD_ID         = NUM_TYPES

# ── Dois cenarios de processo ──────────────────────────────────────────────

SCENARIOS = {
    'fast': dict(
        label='Fast Decay',
        mu    = np.array([0.4, 0.4]),
        alpha = np.array([[0.12, 0.08], [0.08, 0.12]]),
        beta  = 0.50,
        cache_data = 'grid_combined_data_fast.pkl',
        cache_ckpt = 'grid_combined_ckpt_fast.csv',
        data_seed_key = 'grid',
    ),
    'slow': dict(
        label='Slow Decay',
        mu    = np.array([0.3, 0.3]),
        alpha = np.array([[0.008, 0.006], [0.006, 0.008]]),
        beta  = 0.02,
        cache_data = 'grid_combined_data_slow.pkl',
        cache_ckpt = 'grid_combined_ckpt_slow.csv',
        data_seed_key = 'grid_slow',
    ),
}

COR_ROTHP = '#4C72B0'
COR_HOTHP = '#C44E52'

# ── Reproducibilidade ─────────────────────────────────────────────────────

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'

if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3:
            mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

print(f'Device: {device}  |  AMP: {USE_AMP}')
print(f'TRAIN_LENS: {TRAIN_LENS}')
print(f'EXTRAP_FACTORS: {EXTRAP_FACTORS}')
print(f'Seeds: {N_SEEDS}')
print(f'Scenarios: {list(SCENARIOS.keys())}')

In [ ]:
# ============================================================================
# CELULA 3 — Funcoes auxiliares (simulacao, collate, treino, avaliacao)
# ============================================================================

def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(100):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9:
                break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon:
                break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def to_tensors(seqs):
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch, pad_id=PAD_ID):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad = torch.zeros(B, L)
    d_pad = torch.zeros(B, L)
    k_pad = torch.full((B, L), pad_id, dtype=torch.long)
    npm   = torch.zeros(B, L)
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl] = 1.0
        m = causal.clone()
        m[:, sl:] = True
        m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)
    return DataLoader(data, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, generator=g)


config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'Grid',
    'thinning': {'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
                 'patience_counter': 5, 'num_samples_boundary': 5,
                 'dtime_max': 5.0, 'num_step_gen': 1},
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})


def eval_nll(model, dl):
    """Compute NLL over the FULL sequence (not just the extrapolated portion)."""
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast():
                l, n = model.loglike_loss(batch)
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9)


def train_model(cls, train_dl, val_dl, lr, base_seed):
    set_seed(base_seed)
    m = cls(config).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
    scaler = _Scaler(enabled=True) if USE_AMP else None
    best_val = float('inf')
    best_state = None
    no_imp = 0
    for ep in range(EPOCHS):
        m.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast():
                l, n = m.loglike_loss(batch)
                nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()
        v = eval_nll(m, val_dl)
        sched.step(v)
        if v < best_val - 1e-4:
            best_val = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            break
    m.load_state_dict(best_state)
    return m, best_val


def get_eval_bs(test_len):
    if test_len <= 200:
        return 16
    elif test_len <= 500:
        return 8
    elif test_len <= 1000:
        return 4
    elif test_len <= 2000:
        return 2
    else:
        return 1


print('Funcoes prontas.')

In [ ]:
# ============================================================================
# CELULA 4 — Gerar dados para ambos os cenarios (com cache)
# ============================================================================

all_scenario_datasets = {}

for scen_key, scen in SCENARIOS.items():
    mu, alpha, beta = scen['mu'], scen['alpha'], scen['beta']
    cache_path = scen['cache_data']

    if os.path.exists(cache_path):
        print(f'[{scen["label"]}] Cache encontrado: {cache_path}. Carregando...')
        with open(cache_path, 'rb') as f:
            all_scenario_datasets[scen_key] = pickle.load(f)
        print(f'  TRAIN_LENS disponiveis: {list(all_scenario_datasets[scen_key].keys())}')
    else:
        print(f'[{scen["label"]}] Gerando dados do zero...')
        rng = np.random.default_rng(run_seed(scen['data_seed_key'], 'data'))
        datasets = {}

        for tl in TRAIN_LENS:
            print(f'  TRAIN_LEN={tl}...')
            raw_train = [simulate_hawkes(rng, mu, alpha, beta, 50.0 * (tl/50), 5, tl)
                         for _ in range(500)]
            raw_val   = [simulate_hawkes(rng, mu, alpha, beta, 50.0 * (tl/50), 5, tl)
                         for _ in range(150)]
            raw_short = [simulate_hawkes(rng, mu, alpha, beta, 50.0 * (tl/50), 5, tl)
                         for _ in range(200)]

            ds = {
                'train': to_tensors(raw_train),
                'val':   to_tensors(raw_val),
                'short': to_tensors(raw_short),
            }

            for f in EXTRAP_FACTORS:
                target_len = tl * f
                raw_ext = [simulate_hawkes(rng, mu, alpha, beta,
                                           50.0 * (target_len/50),
                                           tl + 5, target_len)
                           for _ in range(200)]
                ds[f'extrap_{f}x'] = to_tensors(raw_ext)
                mean_len = np.mean([len(s['time_seqs']) for s in ds[f'extrap_{f}x']])
                print(f'    {f}x: target={target_len}, mean_len={mean_len:.0f}')

            datasets[tl] = ds

        with open(cache_path, 'wb') as f:
            pickle.dump(datasets, f)
        all_scenario_datasets[scen_key] = datasets
        print(f'  Dados salvos em {cache_path}')

print('\nDados prontos para ambos os cenarios.')

In [ ]:
# ============================================================================
# CELULA 5 — Grid search para ambos os cenarios (com checkpoint)
# ============================================================================

seeds = [BASE_SEED + i * 100 for i in range(N_SEEDS)]

all_scenario_results = {}

for scen_key, scen in SCENARIOS.items():
    print(f'\n{"="*80}')
    print(f'CENARIO: {scen["label"]} (beta={scen["beta"]})')
    print(f'{"="*80}')

    all_datasets = all_scenario_datasets[scen_key]
    ckpt_file = scen['cache_ckpt']

    if os.path.exists(ckpt_file):
        df_done = pd.read_csv(ckpt_file)
        all_results = df_done.to_dict('records')
        done_keys = set(zip(df_done['train_len'], df_done['factor'], df_done['seed']))
        print(f'Checkpoint: {len(all_results)} resultados ja computados.')
    else:
        all_results = []
        done_keys = set()
        print('Nenhum checkpoint. Comecando do zero.')

    total = len(TRAIN_LENS) * N_SEEDS
    count = 0

    for tl in TRAIN_LENS:
        ds = all_datasets[tl]

        tl_remaining = [(seed, f) for seed in seeds for f in EXTRAP_FACTORS
                        if (tl, f, seed) not in done_keys]
        if len(tl_remaining) == 0:
            print(f'TRAIN_LEN={tl}: ja computado, pulando.')
            count += N_SEEDS
            continue

        val_dl   = make_loader(ds['val'],   64)
        short_dl = make_loader(ds['short'], 64)
        ext_dls = {f: make_loader(ds[f'extrap_{f}x'], get_eval_bs(tl * f))
                   for f in EXTRAP_FACTORS}

        for seed_idx, seed in enumerate(seeds):
            seed_remaining = [f for f in EXTRAP_FACTORS if (tl, f, seed) not in done_keys]
            if len(seed_remaining) == 0:
                count += 1
                continue

            count += 1
            print(f'[{count}/{total}] TRAIN_LEN={tl}, Seed {seed_idx+1}/{N_SEEDS}', end='  ')

            train_dl = make_loader(ds['train'], 64, shuffle=True,
                                   seed=run_seed(scen_key, tl, seed))

            rothp, rv = train_model(RoTHP, train_dl, val_dl, lr=1e-3,
                                    base_seed=run_seed(scen_key, tl, 'rothp', seed))
            hothp, hv = train_model(HoTHP, train_dl, val_dl, lr=5e-4,
                                    base_seed=run_seed(scen_key, tl, 'hothp', seed))

            r_short = eval_nll(rothp, short_dl)
            h_short = eval_nll(hothp, short_dl)

            print(f'val: R={rv:.4f} H={hv:.4f}')

            for f in EXTRAP_FACTORS:
                if (tl, f, seed) in done_keys:
                    continue

                test_len = tl * f
                try:
                    r_full = eval_nll(rothp, ext_dls[f])
                    h_full = eval_nll(hothp, ext_dls[f])
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower():
                        print(f'    OOM em TRAIN_LEN={tl} x{f}, pulando.')
                        torch.cuda.empty_cache()
                        gc.collect()
                        r_full = float('nan')
                        h_full = float('nan')
                    else:
                        raise

                all_results.append({
                    'scenario':  scen_key,
                    'train_len': tl,
                    'factor':    f,
                    'test_len':  test_len,
                    'seed':      seed,
                    'r_short':   r_short,
                    'h_short':   h_short,
                    'r_full':    r_full,
                    'h_full':    h_full,
                    'adv':       r_full - h_full,
                    'r_val':     rv,
                    'h_val':     hv,
                })

            del rothp, hothp
            torch.cuda.empty_cache()
            gc.collect()

        df_tmp = pd.DataFrame(all_results)
        df_tmp.to_csv(ckpt_file, index=False)
        print(f'  -> Checkpoint salvo: {len(all_results)} resultados em {ckpt_file}')

    all_scenario_results[scen_key] = pd.DataFrame(all_results)
    print(f'{scen["label"]}: {len(all_results)} resultados totais.')

# Concatena tudo num unico DataFrame
df_all = pd.concat(all_scenario_results.values(), ignore_index=True)
print(f'\nTotal geral: {len(df_all)} resultados.')

In [ ]:
# ============================================================================
# CELULA 6 — Tabela resumo com Wilcoxon signed-rank test
# ============================================================================

def wilcoxon_test(r_vals, h_vals):
    """Wilcoxon signed-rank test (one-sided: HoTHP < RoTHP in NLL).
    
    Tests if HoTHP NLL is significantly lower than RoTHP NLL.
    Returns (statistic, p_value_one_sided).
    """
    diffs = r_vals - h_vals  # positive = HoTHP better
    # If all diffs are zero, no difference
    if np.all(diffs == 0):
        return 0.0, 1.0
    try:
        stat, p2 = wilcoxon(diffs, alternative='greater')
        return stat, p2
    except ValueError:
        # Too few samples or all zeros
        return 0.0, 1.0


def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '~'
    return 'ns'


for scen_key, scen in SCENARIOS.items():
    df = df_all[df_all['scenario'] == scen_key]
    print(f'\n{"="*95}')
    print(f'CENARIO: {scen["label"]} (beta={scen["beta"]})')
    print(f'Wilcoxon signed-rank test (one-sided: HoTHP < RoTHP)')
    print(f'{"="*95}')

    for tl in TRAIN_LENS:
        sub = df[df['train_len'] == tl]
        print(f'\nTRAIN_LEN = {tl}')
        print(f'  {"Factor":>6s}  {"Test len":>8s}  {"NLL RoTHP":>14s}  {"NLL HoTHP":>14s}  '
              f'{"Advantage":>14s}  {"W":>7s}  {"p":>7s}  Sig')
        print(f'  {"-"*6}  {"-"*8}  {"-"*14}  {"-"*14}  {"-"*14}  {"-"*7}  {"-"*7}  ---')

        for f in EXTRAP_FACTORS:
            rows = sub[sub['factor'] == f]
            r_vals = rows['r_full'].values
            h_vals = rows['h_full'].values
            rm, rs = r_vals.mean(), r_vals.std()
            hm, hs = h_vals.mean(), h_vals.std()
            adv = r_vals - h_vals

            w_stat, w_p = wilcoxon_test(r_vals, h_vals)
            sig = sig_label(w_p)

            print(f'  {f:>5}x  {tl*f:>8d}  {rm:.4f}+/-{rs:.4f}  {hm:.4f}+/-{hs:.4f}  '
                  f'{adv.mean():>+.4f}+/-{adv.std():.4f}  W={w_stat:>4.0f}  p={w_p:.3f}  {sig}')

In [ ]:
# ============================================================================
# CELULA 7 — Tabela no formato do paper (Table 1) com Wilcoxon
#
# Formato: para TRAIN_LEN=50, fatores 1x (in-dist), 2x, 5x, 10x
# ============================================================================

PAPER_TL = 50  # TRAIN_LEN usado no paper
PAPER_FACTORS = [2, 5, 10]

print('\n' + '='*100)
print('TABLE 1 (Paper format) — NLL (mean +/- std, N=5 seeds)')
print('Wilcoxon signed-rank test | Bold = best per column per scenario')
print('='*100)

table_data = []

for scen_key, scen in SCENARIOS.items():
    df = df_all[df_all['scenario'] == scen_key]
    sub = df[df['train_len'] == PAPER_TL]

    # 1x (in-distribution) from short test set
    r_short_vals = sub.drop_duplicates('seed')['r_short'].values
    h_short_vals = sub.drop_duplicates('seed')['h_short'].values
    w_stat_1x, w_p_1x = wilcoxon_test(r_short_vals, h_short_vals)

    row_r = {'scenario': scen['label'], 'model': 'RoTHP',
             '1x_mean': r_short_vals.mean(), '1x_std': r_short_vals.std()}
    row_h = {'scenario': scen['label'], 'model': 'HoTHP',
             '1x_mean': h_short_vals.mean(), '1x_std': h_short_vals.std()}

    wilcoxon_results = {'1x': (w_stat_1x, w_p_1x)}

    for f in PAPER_FACTORS:
        rows = sub[sub['factor'] == f]
        r_vals = rows['r_full'].values
        h_vals = rows['h_full'].values
        w_stat, w_p = wilcoxon_test(r_vals, h_vals)

        row_r[f'{f}x_mean'] = r_vals.mean()
        row_r[f'{f}x_std']  = r_vals.std()
        row_h[f'{f}x_mean'] = h_vals.mean()
        row_h[f'{f}x_std']  = h_vals.std()
        wilcoxon_results[f'{f}x'] = (w_stat, w_p)

    table_data.append((scen['label'], row_r, row_h, wilcoxon_results))

# Print formatted
cols = ['1x', '2x', '5x', '10x']
print(f'\n{"Process":<12s} {"Model":<8s}', end='')
for c in cols:
    print(f'  {c:>18s}', end='')
print()
print('-' * 100)

for label, row_r, row_h, w_results in table_data:
    for row in [row_r, row_h]:
        scen_label = label if row == row_r else ''
        print(f'{scen_label:<12s} {row["model"]:<8s}', end='')
        for c in cols:
            m = row[f'{c}_mean']
            s = row[f'{c}_std']
            # Determine if this is the best model for this column
            other = row_h if row == row_r else row_r
            is_best = m < other[f'{c}_mean']
            marker = ' *' if is_best else '  '
            print(f'  {m:.3f}+/-{s:.3f}{marker}', end='')
        print()

    # Print Wilcoxon results for this scenario
    print(f'{"":12s} {"Wilcoxon":8s}', end='')
    for c in cols:
        w_stat, w_p = w_results[c]
        sig = sig_label(w_p)
        print(f'  {"p=":>4s}{w_p:.3f} {sig:>3s}    ', end='')
    print()
    print('-' * 100)

In [ ]:
# ============================================================================
# CELULA 8 — Gera LaTeX para Table 1 (pronto para colar no paper)
# ============================================================================

print('% LaTeX table — cole diretamente no paper')
print(r'\begin{table}[t]')
print(r'\centering')
print(r'\caption{NLL (mean $\pm$ standard deviation, $N=5$ seeds) for the slow and fast decay scenarios. '
      r'Values in \textbf{bold} indicate the best result in each column per process group. '
      r'Lower values are better. Statistical significance assessed via Wilcoxon signed-rank test '
      r'($^{*}$\,$p<0.05$, $^{**}$\,$p<0.01$, $^{***}$\,$p<0.001$).}')
print(r'\label{tab:main_results}')
print(r'\begin{tabular}{llcccc}')
print(r'\toprule')
print(r'Process & Model & $1\times$ & $2\times$ & $5\times$ & $10\times$ \\')
print(r'\midrule')

for idx, (label, row_r, row_h, w_results) in enumerate(table_data):
    latex_label = 'Slow' if 'Slow' in label else 'Fast'
    print(f'\\multirow{{2}}{{*}}{{{latex_label}}}', end='')

    for row_idx, row in enumerate([row_r, row_h]):
        if row_idx > 0:
            print(' ', end='')
        print(f' & {row["model"]}', end='')

        for c in ['1x', '2x', '5x', '10x']:
            m = row[f'{c}_mean']
            s = row[f'{c}_std']
            other = row_h if row == row_r else row_r
            is_best = m < other[f'{c}_mean']

            val_str = f'{m:.3f} $\\pm$ {s:.3f}'
            if is_best:
                val_str = f'\\textbf{{{val_str}}}'
            print(f' & {val_str}', end='')
        print(r' \\')

    if idx < len(table_data) - 1:
        print(r'\midrule')

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

In [ ]:
# ============================================================================
# CELULA 9 — Heatmaps lado a lado (Fast vs Slow)
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (scen_key, scen) in zip(axes, SCENARIOS.items()):
    df = df_all[df_all['scenario'] == scen_key]

    adv_matrix = np.zeros((len(TRAIN_LENS), len(EXTRAP_FACTORS)))
    sig_matrix = np.empty((len(TRAIN_LENS), len(EXTRAP_FACTORS)), dtype=object)

    for i, tl in enumerate(TRAIN_LENS):
        for j, f in enumerate(EXTRAP_FACTORS):
            rows = df[(df['train_len'] == tl) & (df['factor'] == f)]
            r_vals = rows['r_full'].values
            h_vals = rows['h_full'].values
            adv_matrix[i, j] = (r_vals - h_vals).mean()

            _, w_p = wilcoxon_test(r_vals, h_vals)
            sig_matrix[i, j] = sig_label(w_p) if w_p < 0.05 else ''

    annot = np.empty_like(adv_matrix, dtype=object)
    for i in range(adv_matrix.shape[0]):
        for j in range(adv_matrix.shape[1]):
            annot[i, j] = f'{adv_matrix[i,j]:+.3f}\n{sig_matrix[i,j]}'

    # Use same color scale for both heatmaps
    vmax = max(abs(adv_matrix.min()), abs(adv_matrix.max()), 0.1)

    sns.heatmap(adv_matrix, annot=annot, fmt='',
                xticklabels=[f'{f}x' for f in EXTRAP_FACTORS],
                yticklabels=[f'L={tl}' for tl in TRAIN_LENS],
                cmap='RdYlGn', center=0, linewidths=1, linecolor='white',
                vmin=-vmax, vmax=vmax,
                cbar_kws={'label': 'HoTHP advantage (nats)'},
                ax=ax)

    ax.set_xlabel('Extrapolation factor', fontsize=12)
    ax.set_ylabel('TRAIN_LEN (training events)', fontsize=12)
    ax.set_title(f'{scen["label"]} ($\\beta$={scen["beta"]})\n'
                 f'Green = HoTHP better | Wilcoxon test',
                 fontsize=12, fontweight='bold')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('grid_heatmap_combined.pdf', dpi=150, bbox_inches='tight')
plt.savefig('grid_heatmap_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: grid_heatmap_combined.pdf / .png')

In [ ]:
# ============================================================================
# CELULA 10 — Graficos de linhas NLL (lado a lado por cenario)
# ============================================================================

for scen_key, scen in SCENARIOS.items():
    df = df_all[df_all['scenario'] == scen_key]

    fig, axes = plt.subplots(1, len(TRAIN_LENS), figsize=(5 * len(TRAIN_LENS), 5))
    x = np.arange(len(EXTRAP_FACTORS))

    for ax, tl in zip(axes, TRAIN_LENS):
        sub = df[df['train_len'] == tl]

        for col, label, color in [
            ('r_full', 'RoTHP', COR_ROTHP),
            ('h_full', 'HoTHP', COR_HOTHP),
        ]:
            means, stds = [], []
            for f in EXTRAP_FACTORS:
                vals = sub[sub['factor'] == f][col].values
                means.append(vals.mean())
                stds.append(vals.std())
            means = np.array(means)
            stds = np.array(stds)

            ax.plot(x, means, 'o-', color=color, lw=2, ms=7, label=label)
            ax.fill_between(x, means - stds, means + stds, color=color, alpha=0.15)

        # NLL in-dist reference
        r_short_mean = sub.drop_duplicates('seed')['r_short'].mean()
        h_short_mean = sub.drop_duplicates('seed')['h_short'].mean()
        ax.axhline(r_short_mean, color=COR_ROTHP, ls=':', lw=1, alpha=0.5)
        ax.axhline(h_short_mean, color=COR_HOTHP, ls=':', lw=1, alpha=0.5)

        # Wilcoxon significance markers
        for j, f in enumerate(EXTRAP_FACTORS):
            rows = sub[sub['factor'] == f]
            _, w_p = wilcoxon_test(rows['r_full'].values, rows['h_full'].values)
            if w_p < 0.05:
                ymax = max(rows['r_full'].mean(), rows['h_full'].mean()) + rows['r_full'].std()
                ax.text(j, ymax * 1.02, sig_label(w_p), ha='center', fontsize=9, fontweight='bold')

        ax.set_xticks(x)
        ax.set_xticklabels([f'{f}x\n({tl*f} ev)' for f in EXTRAP_FACTORS], fontsize=10)
        ax.set_xlabel('Extrapolation factor')
        ax.set_ylabel('NLL (nats)')
        ax.set_title(f'TRAIN_LEN = {tl}', fontweight='bold', fontsize=13)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.suptitle(f'NLL by Extrapolation Factor — {scen["label"]} ($\\beta$={scen["beta"]})\n'
                 f'Dotted lines = in-dist NLL | Wilcoxon signed-rank test',
                 fontsize=13, fontweight='bold', y=1.04)
    plt.tight_layout()
    fname = f'grid_nll_lines_{scen_key}'
    plt.savefig(f'{fname}.pdf', dpi=150, bbox_inches='tight')
    plt.savefig(f'{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}.pdf / .png')

In [ ]:
# ============================================================================
# CELULA 11 — Wilcoxon detailed report (todas as combinacoes)
# ============================================================================

print('='*100)
print('WILCOXON SIGNED-RANK TEST — Detailed Report')
print('H0: NLL_RoTHP = NLL_HoTHP  |  H1: NLL_RoTHP > NLL_HoTHP (HoTHP is better)')
print('='*100)

wilcoxon_rows = []

for scen_key, scen in SCENARIOS.items():
    df = df_all[df_all['scenario'] == scen_key]

    for tl in TRAIN_LENS:
        sub = df[df['train_len'] == tl]

        for f in EXTRAP_FACTORS:
            rows = sub[sub['factor'] == f]
            r_vals = rows['r_full'].values
            h_vals = rows['h_full'].values
            diffs = r_vals - h_vals

            w_stat, w_p = wilcoxon_test(r_vals, h_vals)

            wilcoxon_rows.append({
                'Scenario': scen['label'],
                'Train L': tl,
                'Factor': f'{f}x',
                'Test L': tl * f,
                'Mean diff': diffs.mean(),
                'Median diff': np.median(diffs),
                'W stat': w_stat,
                'p-value': w_p,
                'Sig': sig_label(w_p),
                'All HoTHP<RoTHP': 'Yes' if np.all(diffs > 0) else 'No',
            })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
print(df_wilcoxon.to_string(index=False))

# Summary
print(f'\n--- Summary ---')
for scen_key, scen in SCENARIOS.items():
    sub = df_wilcoxon[df_wilcoxon['Scenario'] == scen['label']]
    n_sig = (sub['p-value'] < 0.05).sum()
    n_total = len(sub)
    print(f'{scen["label"]}: {n_sig}/{n_total} comparisons significant at p<0.05')

In [ ]:
# ============================================================================
# CELULA 12 — Main result figure (como no paper: slow + fast side by side)
# ============================================================================

# Usa TRAIN_LEN=50 como no paper
PAPER_TL = 50

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

factors_with_1x = [1] + EXTRAP_FACTORS
x = np.arange(len(factors_with_1x))

for ax, (scen_key, scen) in zip(axes, SCENARIOS.items()):
    df = df_all[(df_all['scenario'] == scen_key) & (df_all['train_len'] == PAPER_TL)]

    for col_short, col_full, label, color in [
        ('r_short', 'r_full', 'RoTHP', COR_ROTHP),
        ('h_short', 'h_full', 'HoTHP', COR_HOTHP),
    ]:
        means, stds = [], []

        # 1x (in-dist)
        vals_1x = df.drop_duplicates('seed')[col_short].values
        means.append(vals_1x.mean())
        stds.append(vals_1x.std())

        # 2x, 5x, 10x
        for f in EXTRAP_FACTORS:
            vals = df[df['factor'] == f][col_full].values
            means.append(vals.mean())
            stds.append(vals.std())

        means = np.array(means)
        stds = np.array(stds)

        ax.errorbar(x, means, yerr=stds, fmt='o-', color=color,
                    lw=2, ms=7, capsize=4, label=label)

    # Wilcoxon markers
    for j, f in enumerate(EXTRAP_FACTORS):
        rows = df[df['factor'] == f]
        _, w_p = wilcoxon_test(rows['r_full'].values, rows['h_full'].values)
        if w_p < 0.05:
            ymax = max(rows['r_full'].mean() + rows['r_full'].std(),
                       rows['h_full'].mean() + rows['h_full'].std())
            ax.text(j + 1, ymax * 1.03, sig_label(w_p),
                    ha='center', fontsize=10, fontweight='bold', color='#333')

    ax.set_xticks(x)
    ax.set_xticklabels([f'{f}x' for f in factors_with_1x], fontsize=11)
    ax.set_xlabel('Extrapolation factor', fontsize=12)
    ax.set_ylabel('NLL (nats)', fontsize=12)
    ax.set_title(f'{scen["label"]}', fontweight='bold', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(f'NLL evolution by test length (TRAIN_LEN={PAPER_TL})\n'
             f'Error bars = $\\pm$1 std over {N_SEEDS} seeds',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('main_result.pdf', dpi=150, bbox_inches='tight')
plt.savefig('main_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: main_result.pdf / .png')

In [ ]:
# ============================================================================
# CELULA 13 — Mapa completo TRAIN_LEN vs TEST_LEN absoluto
# ============================================================================

for scen_key, scen in SCENARIOS.items():
    df = df_all[df_all['scenario'] == scen_key]
    all_test_lens = sorted(df['test_len'].unique())

    print(f'\n{scen["label"]}: TRAIN_LEN vs TEST_LEN absoluto (advantage = NLL_RoTHP - NLL_HoTHP)')
    print(f'  {"":>10s}', end='')
    for test_l in all_test_lens:
        print(f'  {test_l:>6d}', end='')
    print()

    for tl in TRAIN_LENS:
        print(f'  L={tl:>4d}  ', end='')
        for test_l in all_test_lens:
            rows = df[(df['train_len'] == tl) & (df['test_len'] == test_l)]
            if len(rows) == 0:
                print(f'  {"---":>6s}', end='')
            else:
                adv = rows['adv'].mean()
                print(f'  {adv:>+.3f}', end='')
        print()

## Como interpretar

### Heatmaps
- **Verde** = HoTHP melhor (vantagem positiva)
- **Vermelho** = RoTHP melhor (vantagem negativa)
- Estrelas = significancia estatistica via **Wilcoxon signed-rank test**

### Wilcoxon signed-rank test
- Teste nao-parametrico pareado (mais robusto que t-test com N=5)
- H0: NLL_RoTHP = NLL_HoTHP
- H1: NLL_RoTHP > NLL_HoTHP (HoTHP tem NLL menor = melhor)
- `*` p<0.05, `**` p<0.01, `***` p<0.001

### Table 1 (Paper)
- Formato pronto para o paper BRACIS
- TRAIN_LEN=50, fatores 1x/2x/5x/10x
- NLL computada sobre a **sequencia completa**

### Expectativa
- **Fast decay**: HoTHP deveria ter vantagem significativa, crescendo com o fator de extrapolacao
- **Slow decay**: ambos os modelos devem ter desempenho similar (sem vantagem significativa)